# `queso-cluster` Tutorial

**This is a work in progress!!!** 


## A brief introduction
This package is still under active development!! I am working on it.

For package installation:

In [1]:
## For Juypter
%pip install queso-cluster

## For pip
# pip3 install /disk/data/sriley/queso-cluster

Note: you may need to restart the kernel to use updated packages.


There is one additional picece that is required for this version of QuESO: the `eventManager.yml` file. This file will contain the user defined configuration for not just the clustering but also extra bits of data calibration. An entry looks like

```yaml
- event:
    date: '2023-05-03'
    label: 'Strong flare signatures'
    run: 
      - label: 'timeDependent'
        approach: 'evo'
        overwrite: True
        alignment_dir: "aligned_x5bin"
        config: {frames: [-1, 3, 1],
              keepI0: [3],
              bbox: [0, 125, 1188, 1629],
              primary: {src: 'AEVEG-SPD', S1: [1, 5, 2, 4, 1], lines: {'AEVEG-SPD': ['main']}}, 
        }
    src: 
      - id: {'instrument': 'ViSP', 'data': 'AEVEG', mod: "SPD"}
        theme: '#527A00'
        residual: False
        clustering:
          S0: 
            - {label: 'window', layerConfig: {'bins': [0, 0.52, 0.60, 2]}}
          S1: 
            - {layerGroups: 5, layerConfig: {'converge': 1e-6, ss_thresh: [0, 0, 0, 0, 0, 0]}}
        axis_fit: {coeff: [-2.81754321e-8  1.91621915e-3  8.53133874e+2], unit: "angstrom"}
        lines:
          - {label: 'main', window: [462, 568], continuum: 26, core: 515}
      - id: {'instrument': 'ViSP', 'data': 'BZNNG', mod: "SPD"}
        theme: {'D1': '#0000FF', 'D2': '#FF0000', 'Ni': '#FF0000'}
        residual: True
        axis_fit: {coeff: [-1.60283877e-8  1.42050858e-3  5.88875195e+2], unit: "angstrom"}
        lines:
          - {label: 'D2', continuum: 375, core: 86,  window: [51, 121]}
          - {label: 'D1', continuum: 375, core: 509, window: [473, 545]}
          - {label: 'Ni', continuum: 375, core: 291, window: [256, 327]}
```

### Getting started

To get started, we are going to import the package (note the difference of the **dash** and **underscore** in the package name). And load in some file paths. 

In [2]:
import queso_cluster as qc
import os

#os.environ["QUESO-DAT"] = '/disk/data/DKIST/20230503'

One last note, the figure functions output a `matplotlib` figure object.  

### Instrument Loader Objects

To handle the specific physical attributes of the data, we have developed a framework to import only the essentials. Currently, we only have a built-in object for ViSP. The instrument object will hold the dataset compressed into a `dask` array for lazy loading to minimize memory usage during some of the calculations.  

## Time-independent clustering

This is just fancy words for the standard approach to cluster the evolution of spectra. As it suggests, this clustering approach doesn't use any temporal information during the clustering. See [Udeas et al., 2026]() for an example.

The basic principle of using this approach is to cluster the entire dataset then afterwards reintroduce the temporal information to get the sequence of spectral profiles associated with the feature. This method does work, but there are some considerations that need to be taken. 

1. **Feature density** or the number of resolved clustered within the coordinate space. In clustering, a high feature density is actually bad. The more clusters you need the more the boundaries of the clusters is blurred. While there is not a specific threshold for feature density, a smaller "global" feature density is preferred.
2. **

This short tutorial will go through the steps to use the `ti` sub-module with a time series spectral dataset.  

In [3]:
import numpy as np
from queso_cluster import ti
from queso_cluster.runners import base as runBase
from queso_cluster.atoms import norm as normAtom
from queso_cluster.loaders.visp import visp

def main(config):
	#> Note: This creates the instrument object for the ViSP observations
	ViSPobj = visp("/disk/data/DKIST/20250503")
	ViSPobj.load()

	#> Note: Here we are going to take only the first four frames and flatten them. The clustering functions only accept 2D arrays.
	ViSPobj.dataSquare = ViSPobj[:4, ...].reshape((vispObj.shape[1]*4, ViSPobj.shape[-1]))

	#> Note: This function imports everything from the instrument object and has the specific clustering functions that we will use for the time independent mode.
	tiObj = ti.timeIndependent(config, config.runners.label, ViSPobj)

	#> Note: This preps the data. Specifically we want to normalize by the continuum intensity
	prepSquare = runBase.runPrep(tiObj.dataSquare,
									norm=normAtom.normContinuum, 
									continuumIndx=tiObj.continuum)

	#> Note: We are most interested in the flaring pixels. These are very bright compared with the surrounding area, 
	maskLine = np.ones(prepSquare.shape[0]).astype(bool)
	keepI0 = None
	if "keepI0" in list(config.runners.config.keys()):
		keepI0 = config.runners.config['keepI0']
	
	#> Note: This is where the magic happens. This function will handle calls to the base clustering functions which handle the calculations.
	#> Note: The output in a 1D flattened array containing the labels for each pixel and a data object which stores the quality metrics of the cluster.
	labelLine, scoreTuple,  = tiObj.cluster(prepSquare, maskLine, 
											keepI0=keepI0, kLst=config.srcLst.clusterConfig['optimized'])

	return(labelLine, scoreTuple)


/nfs/hl0/data/sriley/src/sriley/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from queso_cluster.loaders.event import eventInput
eventManager = eventInput("./eventManager.yml", 1, 0)


epochDev, labelLine = main(eventManager.event)


ImportError: cannot import name 'eventInput' from 'queso_cluster.loaders.event' (/nfs/hl0/data/sriley/src/sriley/lib/python3.11/site-packages/queso_cluster/loaders/event.py)

## Time-dependent clustering
Here is where we enter more experimental waters. The main objective is to try and use the time information inside the algorithm where each cluster is a sequence of spectral profiles. One example of this was [presented at the Solar Physics Division conference in 2025](https://sriley.dev/), where we clustered each frame independently then glued the results together at the end. The benefit is a smaller number of clusters is needed per time step due to a smaller diversity of spectral features relative to the entire set of frames. 

This section of the tutorial will discuss the `td` sub-module with the same time series spectral dataset as before. 

In [ ]:
import numpy as np
from queso_cluster import td
# from queso_cluster.runners import base as runBase
# from queso_cluster.atoms import norm as normAtom
from queso_cluster.loaders.visp import visp

def main(config):
	#> detail: 
	#> param type config:
	#> return (type): 
	#> test-method:

	# srcUse = config.runners.config['primary']['src']
	# srcConfigPrimary 	= config.srcLst[config.srcLabelLst.index(srcUse)]
	# config.srcLst = srcConfigPrimary

	ViSPobj = visp('/disk/data/DKIST/20230503/AEVEG/')
	ViSPobj.load()

	evoDev = td.timeDependent(config, config.runners.label, ViSPobj)

	#> Note: Creates a mask for data within a specific coordinate range
	bboxMask = np.zeros((evoDev.rasterSize, evoDev.alongSlitSize))
	bEx = config.runners.config['bbox']
	bboxMask[bEx[0]:bEx[1], bEx[2]:bEx[3]] = 1
	bboxMask = bboxMask.reshape(bboxMask.shape[0]*bboxMask.shape[1])

	#> Note: Here I know that there is a bright feature at time step 2. Then, I want four frames centered on that peak -2, -1, +1, +2
	timeFrames, tLst = evoDev.timeFrames(peakTime=2, nframes=4)
	
	#> Note: As before, we are generally more interested in the brighter features so we will opt to mask out the lower intrinsic bins
	#> Note: In this case, we will use the locations of bright pixels in the peak frame to focus the clustering.
	keepI0 = None
	if "keepI0" in list(config.runners.config.keys()):
		keepI0 = config.runners.config['keepI0']
	intrinsicLine = base._mainIntrinsic(config.srcLst, 
										np.floor(timeFrames[1, ...]*100)/100., 0, intrinsicSkip=False)
	intrinsicLine = auxAtom.pick_jth_label(intrinsicLine, 0).astype(int)

	#> Note: This combines both the coordinate mask and the intrinsic bin mask
	maskLine = bboxMask.astype(bool)#np.ones(prepSquare.shape[0]).astype(bool)
	if not (keepI0 is None):
		i0Mask = np.zeros(timeFrames.shape[1], dtype=bool)
		for i in keepI0:
			i0Mask[(intrinsicLine == i)] = 1
		maskLine *= i0Mask

	#> Note: This is our list of cluster groups for each time step
	klst = config.runners.config['primary']['S1']
	#> Note: This is a specific fucntion which prepares the whole time sequence 
	prepCube = evoDev.prepSequence(timeFrames, norm=normAtom.normContinuum, continuumIndx=self.continuum)
	#> Note: Now the magic happens. Right now this will simply cluster each frame seperately then combine the labels to form a sequence. 
	labelSquare = evoDev.clusterSequence(prepCube, maskLine, klst, intrinsicLine=intrinsicLine)
	return(evoDev, labelSquare)

In [ ]:
from queso_cluster.loaders.event import eventInput
eventManager = eventInput("./eventManager.yml", 1, 0)


epochDev, labelLine = main(eventManager.event)